# Araras: Gemma 4 copilot for Brazilian rare-disease care

## Runnable demo of the end-to-end pipeline

**Pipeline**: `PT-BR free text → araras-hpo-brasil (BioLORD fine-tune) → araras-gemma4-e4b (Gemma 4 SFT) → canonical ORPHA lookup → PCDT overlay → SUS conduta`

**Benchmark result on RareBench-BR L5_realsus** (DataSUS-anchored, 52,343 real APAC trajectories):
- R@1 strict ORPHA = **87.5%**
- Track B PCDT-correct medication = **100% (22/22)**

**Companion repos**:
- 🤗 https://huggingface.co/Raras-AI/araras-gemma4-e4b-v4-gguf (230+ dl)
- 🐙 https://github.com/rarasAI/araras-gemma4
- 🐙 https://github.com/rarasAI/rarebench-br

*Submission to the Gemma 4 Good Hackathon — built by Dimas, paciente raro, 20 anos pro diagnóstico.*

## 1. Setup — install deps + clone repos

In [ ]:
!pip install -q huggingface_hub sentence-transformers numpy requests llama-cpp-python
!git clone -q https://github.com/rarasAI/araras-gemma4 || (cd araras-gemma4 && git pull -q)
!git clone -q https://github.com/rarasAI/rarebench-br || (cd rarebench-br && git pull -q)

## 2. Download Gemma 4 GGUF (5.3 GB) — Q4_K_M quantization for edge inference

In [ ]:
from huggingface_hub import hf_hub_download
import os
MODEL_PATH = hf_hub_download(
    repo_id='Raras-AI/araras-gemma4-e4b-v4-gguf',
    filename='araras-gemma4-e4b-v4-Q4_K_M.gguf',
)
print(f'Model size: {os.path.getsize(MODEL_PATH)/1e9:.2f} GB at {MODEL_PATH}')

## 3. Load Gemma 4 via llama-cpp-python (GPU if available, else CPU)

In [ ]:
from llama_cpp import Llama
llm = Llama(
    model_path=MODEL_PATH,
    n_ctx=8192,
    n_gpu_layers=-1,  # full offload to GPU if available
    verbose=False,
)
print('Gemma 4 E4B loaded')

## 4. Load araras-hpo-brasil — BioLORD fine-tune for PT-BR clinical → HPO

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np, json
hpo = SentenceTransformer('Raras-AI/araras-hpo-brasil')
print('araras-hpo-brasil loaded')

# Test PT-BR clinical idiom
queries = ['amarelão', 'bebê molinho', 'fácies grosseira', 'hepatoesplenomegalia']
embs = hpo.encode(queries, normalize_embeddings=True)
print(f'Embeddings shape: {embs.shape}')
for q, e in zip(queries, embs):
    print(f'  {q!r:30}  embedding[:4]={e[:4]}')

## 5. The full pipeline — query a clinical case end-to-end

In [ ]:
import sys
sys.path.insert(0, 'araras-gemma4/pipeline')
from araras_pipeline import resolve_orpha, sus_conduta, CANONICAL

def araras_inference(case_text):
    """Pipeline: Gemma 4 → ORPHA lookup → PCDT overlay."""
    sys_prompt = """Você é ARARAS, copiloto clínico de doenças raras em PT-BR.
Liste o TOP-5 diferencial em ordem de probabilidade. Formato (uma linha por dx):
1. Nome — Justificativa
NUNCA invente código ORPHA — escreva apenas o nome canônico da doença em PT-BR."""
    
    resp = llm.create_chat_completion(
        messages=[
            {'role': 'system', 'content': sys_prompt},
            {'role': 'user', 'content': case_text},
        ],
        temperature=1.0, top_p=0.95, top_k=64, repeat_penalty=1.15,
        max_tokens=600,
    )
    raw = resp['choices'][0]['message']['content']
    
    # Stage 3 + 4: ORPHA lookup + PCDT overlay
    orphas = resolve_orpha(raw)
    conduta = sus_conduta(orphas[0]) if orphas else None
    
    return {'gemma_raw': raw, 'orphas': orphas, 'sus_conduta': conduta}

# Test: classic MPS case
case = ('Menina 8 anos, hepatoesplenomegalia progressiva, opacidade corneana bilateral, '
        'atraso cognitivo importante, fácies grosseira, irmã mais velha com mesmo quadro.')

print('=== INPUT ===')
print(case)
result = araras_inference(case)
print('\n=== Gemma 4 raw output ===')
print(result['gemma_raw'][:500])
print('\n=== Resolved ORPHAs ===')
for o in result['orphas'][:3]:
    print(f"  {o['orpha']} (matched {o['matched_keyword']!r}, PCDT={o['pcdt_slug']})")
print('\n=== SUS Conduta ===')
print(result['sus_conduta'])

## 6. Run on a small sample of RareBench-BR L5_realsus

In [ ]:
cases = [json.loads(l) for l in open('rarebench-br/cases/L5_realsus.jsonl')]
print(f'L5_realsus has {len(cases)} cases across {len(set(c["ground_truth"]["primary_orphanet"] for c in cases))} diseases')
print()

# Quick demo on first case per disease (12 cases)
import random
random.seed(42)
by_d = {}
for c in cases:
    by_d.setdefault(c['ground_truth']['primary_orphanet'], []).append(c)
demo = [random.choice(lst) for lst in by_d.values()]

hits = 0
for i, c in enumerate(demo, 1):
    gold = c['ground_truth']['primary_orphanet']
    r = araras_inference(c['free_text_pt'])
    top1 = r['orphas'][0]['orpha'] if r['orphas'] else None
    ok = top1 == gold
    hits += int(ok)
    mark = '✓' if ok else '✗'
    print(f'  [{i:>2}/{len(demo)}] {c["case_id"]} gold={gold} pred={top1} {mark}')

print(f'\nR@1 on this 12-case demo: {hits}/{len(demo)} = {100*hits/len(demo):.0f}%')
print('Full bench (267 cases) shows R@1 strict = 87.5%, Track B PCDT-correct = 100%')

## 7. What this enables

The same pipeline runs on a Brazilian patient's phone — offline. The model knows:
- Which of the 24 official MS PCDTs applies (gov.br/conitec)
- Which medication CEAF actually dispenses (matched against 52,343 real APAC trajectories)
- Which reference center is closest

All without sending the patient's clinical data to any cloud LLM. LGPD-safe by construction.

### Wired into raras.org production

The [raras.org](https://raras.org) Next.js app already has `/api/copilot` accepting `COPILOT_PRIMARY_LLM=gemma` to route through this stack. The patient registry (Carteira) shares 3,072-dim BioLORD+PhenoGnet+Gemini fused embeddings — the same vectors retrieved for cohort matching feed the Gemma 4 prompt.

### Links
- **Writeup**: [github.com/rarasAI/araras-gemma4/blob/main/WRITEUP.md](https://github.com/rarasAI/araras-gemma4)
- **Pipeline source**: [github.com/rarasAI/araras-gemma4/pipeline](https://github.com/rarasAI/araras-gemma4/tree/main/pipeline)
- **Benchmark**: [github.com/rarasAI/rarebench-br](https://github.com/rarasAI/rarebench-br)
- **Model weights**: [🤗 Raras-AI/araras-gemma4-e4b-v4-gguf](https://huggingface.co/Raras-AI/araras-gemma4-e4b-v4-gguf)
- **Live platform**: [raras.org](https://raras.org)

— Dimas, paciente raro 🇧🇷